# AI Incident Database: model incidents by developer

**Purpose.** This notebook builds `aiid_model_incidents_by_developer.csv`, the table behind view 5b of Story 3 (how often each of the eleven developers appears in reported model incidents, and what kind of incident it was). It reads the weekly Excel export of the AI Incident Database, finds the eleven developers, keeps the incidents that are about a model, adds the MIT risk domain and groups it into four incident types.

**Question served.** Story 3: "how openly each developer builds its models and how often it appears in reported incidents, so that I can tell the committee what is known and unknown about the developers we would rely on."

**How to run.** Run all cells from top to bottom. Nothing needs to be edited. The notebook reuses the export already in `raw/` (see `REFRESH` in Section 3). Two files in `manual/` hold the group's own readings. If a new export contains incidents that these files do not cover, the notebook stops and writes the missing incidents to `processed/` so that someone can read them and add them (see "How to update"). The last section lists the file to copy by hand into the story's `data/` folder.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one Excel file (weekly export), plus two hand-made files in `manual/` |
| Where it comes from | Responsible AI Collaborative, AI Incident Database. Snapshots page https://incidentdatabase.ai/research/snapshots/ |
| Licence and authorisation | Creative Commons Attribution-ShareAlike 4.0 for the database collections, report text excluded. No account or key needed. Details in Section 1 |
| Download size | about 2.6 MB (export of 12 Oct 2026) |
| Expected time | download a few seconds, the whole notebook about 10 to 20 seconds with the file in `raw/` |
| Data as of | `processed/data_currency.csv` (written by the last cells) |
| To update | Read "How to update" at the end. A new export needs new manual readings for the incidents that are new |

## 1. Source, reference links and authorisation

| Item | Detail |
|---|---|
| Publisher | Responsible AI Collaborative |
| Dataset | AI Incident Database, weekly Excel export, sheet `Incidents` (one row per incident) |
| Snapshots page | https://incidentdatabase.ai/research/snapshots/ |
| File used | `AIID_Excel_Export-20261012.xlsx` (export of 12 Oct 2026) |
| Terms of use | https://incidentdatabase.ai/terms-of-use |
| Licence | Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0) for the database collections. The text of reports, submissions, images and videos is not covered. This notebook uses incident fields only (number, date, title, developer, deployer, implicated systems, MIT risk domain and subdomain, report count) and reads the incident description to prepare the manual files. Files published from it carry CC BY-SA 4.0 |
| Authorisation | No account, key or login needed. The file is public. We do not scrape any page |
| MIT risk domain | The columns `Risk Domain` and `Risk Subdomain` come from the MIT AI Risk Repository classification that the database carries. MIT has not classified every incident |
| Credit line for the page | Data: AI Incident Database, Responsible AI Collaborative, CC BY-SA 4.0, export of 12 Oct 2026 |

## 2. Table structure

### 2.1 Source: sheet `Incidents` of the export

The sheet has two title rows above the header, so the notebook finds the header row by the cell `Incident ID`. One row per incident. 1,734 rows and 59 columns in the export of 12 Oct 2026. Columns used:

| Source column | Meaning |
|---|---|
| `Incident ID` | incident number |
| `date` | date the harm occurred (editor resolved) |
| `title` | incident title |
| `description` | incident description (read to prepare the manual files, not saved in the output) |
| `developer` | alleged developer, as typed in the database, several names joined by a comma |
| `deployer` | alleged deployer (read to prepare the manual files) |
| `Implicated Systems` | the AI systems named in the incident, for example "ChatGPT" |
| `report_count` | number of reports about the incident |
| `Risk Domain`, `Risk Subdomain` | MIT risk domain and subdomain. Empty for 238 incidents that MIT has not classified |

### 2.2 Hand-made files in `manual/`

| File | What it holds | Who made it |
|---|---|---|
| `aiid_incident_labels_400.csv` | One row for each incident from 2020 onward that names one of the eleven developers (400 incidents on 12 Oct 2026). `label` is `model` (the incident is about the model or a model-powered assistant or agent), `platform` (a recommendation or non-language-model algorithm), `operations` (not model behaviour) or `venue_only` (the developer's product is only the place where a scam appeared) | Group reading of the incident title, description, developer, deployer and implicated-systems fields, done with AI assistance. A second group member has not yet checked it |
| `aiid_66_assigned_mit_domain.csv` | One row for each model incident that MIT has not classified (66 on 12 Oct 2026). `assigned_domain` is the MIT risk domain that fits best, chosen by following how MIT classified similar incidents. `basis` says what was read and `comparable_MIT_classified_incidents` names the MIT precedent where there is one | Same as above. `second_check` is `pending` for every row |

### 2.3 Output `processed/aiid_model_incidents_by_developer.csv`

One row for each pair of model incident and developer (an incident that names two developers has two rows).

| Output column | Type | How it is made |
|---|---|---|
| `incident_id`, `date`, `year`, `title`, `report_count` | | copied from the export |
| `developer` | text | one of the eleven developers, found by the patterns in Section 6 |
| `found_in` | text | `developer field` or `implicated system only` |
| `mit_risk_domain`, `mit_risk_subdomain` | text | MIT's own classification, empty when MIT has not classified the incident |
| `group_mit` | text | the four-way incident type that MIT's domain maps to, or `No MIT class` |
| `assigned_domain` | text | the domain from `manual/aiid_66_assigned_mit_domain.csv`, only for incidents without an MIT class |
| `type_source` | text | `MIT` or `Project group` |
| `domain_shown` | text | `mit_risk_domain` if MIT classified the incident, otherwise `assigned_domain` |
| `group` | text | the incident type shown on the page, from `domain_shown` |

The four incident types: **Wrong output or failure** (Misinformation, AI system safety), **Data and security** (Privacy & Security), **Misuse by others** (Malicious Actors & Misuse) and **Harm to users or society** (Discrimination and Toxicity, Human-Computer Interaction, Socioeconomic & Environmental Harms).

## 3. Set up the paths and download helper

The notebook writes the file it downloads to `raw/` and the tables it builds to `processed/`. `get_export()` tries a fresh download only when `REFRESH = True`, then the newest export already in `raw/`, then the newest export saved by hand in `manual/`.

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "aiid_model_incidents"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER}. Current folder: {cwd}")

MANUAL = HERE / "manual"                       # files that people prepared or saved by hand
STORY_DATA = HERE.parents[1] / "data"          # the story's own data folder. The notebook never writes here.
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)
print("Story data     :", STORY_DATA, "(copy files here by hand, see the last section)")

HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 300
REFRESH = False  # False: reuse the file already in raw/ (same snapshot as the page). True: try a fresh download first.


def record(source_id: str, url: str, path: Path, how: str, keep_date: bool = False) -> None:
    """Add (or replace) the retrieval-log entry for one raw file. keep_date leaves an entry untouched when the file is unchanged."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    if keep_date and log.get(source_id, {}).get("sha256") == hashlib.sha256(content).hexdigest():
        return  # same file as before: keep the original entry (date, address and how it was obtained)
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")

import re
from urllib.parse import urljoin

import numpy as np

SNAPSHOTS_URL = "https://incidentdatabase.ai/research/snapshots/"
EXPORT_NAME = re.compile(r"AIID_Excel_Export-(\d{8})\.xlsx")
FIRST_YEAR = 2020


def newest_local() -> Path | None:
    files = [p for folder in (RAW, MANUAL) for p in folder.glob("AIID_Excel_Export-*.xlsx")]
    return max(files, key=lambda p: (EXPORT_NAME.search(p.name).group(1), p.parent == RAW), default=None)


def get_export() -> Path:
    if REFRESH:
        try:
            page = requests.get(SNAPSHOTS_URL, headers=HEADERS, timeout=60)
            page.raise_for_status()
            links = {urljoin(SNAPSHOTS_URL, h) for h in re.findall(r'href="([^"]+\.xlsx)"', page.text)
                     if EXPORT_NAME.search(h)}
            assert links, "no Excel export link on the snapshots page"
            url = max(links, key=lambda u: EXPORT_NAME.search(u).group(1))
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path = RAW / EXPORT_NAME.search(url).group(0)
            path.write_bytes(response.content)
            record("aiid_excel_export", url, path, "downloaded")
            return path
        except Exception as error:  # noqa: BLE001 - any failure falls back to the local copy
            print(f"Download of the export failed ({error}). Using the newest local copy.")
    path = newest_local()
    assert path is not None, "No AIID_Excel_Export-YYYYMMDD.xlsx in raw/ or manual/. Save the newest export there."
    if path.parent == MANUAL:
        shutil.copy2(path, RAW / path.name)
        path = RAW / path.name
        record("aiid_excel_export", "saved by hand in manual/", path, "copied from manual folder")
    else:
        record("aiid_excel_export", SNAPSHOTS_URL, path, "reused file in raw/", keep_date=True)
    return path

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\aiid_model_incidents
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\aiid_model_incidents\manual
Story data     : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\data (copy files here by hand, see the last section)


## 4. Acquire the file

In [2]:
raw_file = get_export()
export_date = datetime.strptime(EXPORT_NAME.search(raw_file.name).group(1), "%Y%m%d").date()
log = json.loads(LOG_PATH.read_text(encoding="utf-8"))["aiid_excel_export"]
print("File:", raw_file.name, "| export date:", export_date)
print(json.dumps(log, indent=2))

File: AIID_Excel_Export-20261012.xlsx | export date: 2026-10-12
{
  "url": "https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/AIID_Excel_Export-20261012.xlsx",
  "file": "AIID_Excel_Export-20261012.xlsx",
  "how": "downloaded",
  "retrieved_utc": "2026-10-17T23:10:45Z",
  "bytes": 2556570,
  "sha256": "1fbf053dd14f896ccd8bd2c5357fae5838adf9d00e7144d75b295da12552f8a0"
}


## 5. Inspect

Check the sheets, the header row, the columns and the missing values before changing anything.

In [3]:
sheets = pd.ExcelFile(raw_file).sheet_names
print("Sheets:", sheets)
probe = pd.read_excel(raw_file, sheet_name="Incidents", header=None, nrows=10)
header_row = int(probe.index[probe.iloc[:, 0].astype(str).str.strip().eq("Incident ID")][0])
print("Header row (counting from 0):", header_row)
inc_raw = pd.read_excel(raw_file, sheet_name="Incidents", header=header_row)

needed = ["Incident ID", "date", "title", "description", "developer", "deployer", "Implicated Systems",
          "report_count", "Risk Domain", "Risk Subdomain"]
missing_cols = [c for c in needed if c not in inc_raw.columns]
print("Missing columns:", missing_cols)
assert not missing_cols, "The export changed its columns. Update Section 2 and the names used below."

print()
print("Rows:", len(inc_raw), "| unique Incident ID:", inc_raw["Incident ID"].nunique())
print("Dates:", inc_raw["date"].min(), "to", inc_raw["date"].max())
print("Missing values in the columns used:")
print(inc_raw[needed].isna().sum())
print()
print(inc_raw["Risk Domain"].value_counts(dropna=False))

Sheets: ['Incidents', 'Reports', 'Entities', 'Data Dictionary', 'Coverage Map']
Header row (counting from 0): 2
Missing columns: []

Rows: 1734 | unique Incident ID: 1734
Dates: 1983-09-26 00:00:00 to 2026-10-03 00:00:00
Missing values in the columns used:
Incident ID             0
date                    0
title                   0
description             0
developer               1
deployer                3
Implicated Systems    549
report_count            0
Risk Domain           238
Risk Subdomain        239
dtype: int64

Risk Domain
Malicious Actors & Misuse                      566
AI system safety, failures, and limitations    313
Discrimination and Toxicity                    243
NaN                                            238
Misinformation                                 196
Privacy & Security                             112
Human-Computer Interaction                      42
Socioeconomic & Environmental Harms             24
Name: count, dtype: int64


## 6. Find the eleven developers

Two lists of patterns, both written in lower case and matched with word boundaries, so "Meta" does not match "metadata".

- **Developer field.** Company names as people typed them in `developer`. "Facebook" counts as Meta.
- **Implicated systems.** The same company names and also product names such as ChatGPT, Claude, Gemini, Llama, Grok, Qwen and Kimi.

An incident that names one of the eleven in the developer field uses only those rows. The implicated-systems field is used only for an incident whose developer field names none of the eleven, because in other incidents the field often lists tools that were involved but not at fault (for example search engines and chatbots that appear in a list). Only incidents from 2020 onward are read, because the model era the page is about starts then.

In [4]:
DEVELOPER_PATTERNS = {
    "OpenAI": r"\bopen\s?ai\b",
    "Anthropic": r"\banthropic\b",
    "Google": r"\bgoogle\b|\bdeepmind\b|\balphabet\b",
    "Meta": r"\bmeta\b(?![-\s]?data)|\bfacebook\b",
    "xAI": r"\bx\.?ai\b",
    "Mistral AI": r"\bmistral\b",
    "DeepSeek": r"\bdeep[\s-]?seek\b",
    "Alibaba": r"\balibaba\b|\bqwen\b|\btongyi\b",
    "Z.ai (Zhipu)": r"\bzhipu\b|\bz\.ai\b|\bchatglm\b",
    "Moonshot": r"\bmoonshot\b|\bkimi\b",
    "MiniMax": r"\bminimax\b|\bhailuo\b",
}
SYSTEM_PATTERNS = {
    "OpenAI": r"\bopen\s?ai\b|\bchat\s?gpt\b|\bgpt[\s-]?\d|\bdall[\s·-]?e\b|\bsora\b|\bcodex\b|\bwhisper\b",
    "Anthropic": r"\banthropic\b|\bclaude\b",
    "Google": r"\bgoogle\b|\bgemini\b|\bbard\b|\bdeepmind\b|\bveo\b|\bimagen\b",
    "Meta": r"\bmeta\b(?![-\s]?data)|\bllama\b",
    "xAI": r"\bx\.?ai\b|\bgrok\b",
    "Mistral AI": r"\bmistral\b|\ble chat\b",
    "DeepSeek": r"\bdeep[\s-]?seek\b",
    "Alibaba": r"\balibaba\b|\bqwen\b|\btongyi\b",
    "Z.ai (Zhipu)": r"\bzhipu\b|\bz\.ai\b|\bchatglm\b|\bglm[\s-]?\d",
    "Moonshot": r"\bmoonshot\b|\bkimi\b",
    "MiniMax": r"\bminimax\b|\bhailuo\b|\btalkie\b",
}


def mentions(text, patterns: dict[str, str]) -> set[str]:
    text = str(text).lower() if pd.notna(text) else ""
    return {dev for dev, pattern in patterns.items() if re.search(pattern, text)}


inc = inc_raw.rename(columns={"Incident ID": "incident_id", "Risk Domain": "mit_risk_domain",
                              "Risk Subdomain": "mit_risk_subdomain",
                              "Implicated Systems": "implicated_systems"}).copy()
inc["date"] = pd.to_datetime(inc["date"], errors="coerce")
inc = inc.dropna(subset=["incident_id", "date"]).drop_duplicates("incident_id")
inc["incident_id"] = inc["incident_id"].astype(int)
inc["year"] = inc["date"].dt.year
inc = inc[inc["year"] >= FIRST_YEAR]
print(f"Incidents from {FIRST_YEAR}: {len(inc):,}")

rows = []
for r in inc.itertuples(index=False):
    in_developer = mentions(r.developer, DEVELOPER_PATTERNS)
    in_systems = mentions(r.implicated_systems, SYSTEM_PATTERNS)
    chosen = in_developer if in_developer else in_systems
    for dev in sorted(chosen):
        rows.append({"incident_id": r.incident_id, "developer": dev,
                     "found_in": "developer field" if in_developer else "implicated system only"})
mentioned = pd.DataFrame(rows)
by_incident = mentioned.groupby("incident_id")["found_in"].first()
print("Incidents that name at least one of the eleven:", mentioned["incident_id"].nunique())
print(by_incident.value_counts().to_string())

Incidents from 2020: 1,481
Incidents that name at least one of the eleven: 400
found_in
developer field           378
implicated system only     22


## 7. Manual layer 1: is the incident about a model?

The page is about model developers. Some incidents that name one of the eleven developers are not about a model. Examples: a recommendation algorithm on a social platform, a data-centre or business-operations matter, or a scam that only appeared on a developer's platform. The file `manual/aiid_incident_labels_400.csv` holds the group's label for each incident. The notebook stops if an incident that names one of the eleven has no label, writes those incidents to `processed/needs_label.csv` and asks for a reading.

In [5]:
labels = pd.read_csv(MANUAL / "aiid_incident_labels_400.csv")
assert labels["incident_id"].is_unique, "duplicated incident number in aiid_incident_labels_400.csv"
assert set(labels["label"]) <= {"model", "platform", "operations", "venue_only"}, set(labels["label"])

named_ids = set(mentioned["incident_id"])
no_label = sorted(named_ids - set(labels["incident_id"]))
if no_label:
    (inc[inc["incident_id"].isin(no_label)][["incident_id", "date", "title", "description", "developer", "deployer",
                                             "implicated_systems"]]
     .to_csv(OUT / "needs_label.csv", index=False, encoding="utf-8"))
    raise SystemExit(f"{len(no_label)} incidents name one of the eleven but have no label. "
                     f"Read them in processed/needs_label.csv and add rows to manual/aiid_incident_labels_400.csv.")
print("All", len(named_ids), "incidents that name one of the eleven have a label.")
print(labels[labels["incident_id"].isin(named_ids)]["label"].value_counts().to_string())

All 400 incidents that name one of the eleven have a label.
label
model         318
platform       61
venue_only     16
operations      5


## 8. Keep the model incidents and add the MIT risk domain

Only incidents labelled `model` are kept. The MIT risk domain is mapped to the four incident types shown on the page. Where MIT has not classified the incident, `group_mit` is `No MIT class`.

In [6]:
GROUP_OF_DOMAIN = {
    "Misinformation": "Wrong output or failure",
    "AI system safety, failures, and limitations": "Wrong output or failure",
    "Privacy & Security": "Data and security",
    "Malicious Actors & Misuse": "Misuse by others",
    "Discrimination and Toxicity": "Harm to users or society",
    "Human-Computer Interaction": "Harm to users or society",
    "Socioeconomic & Environmental Harms": "Harm to users or society",
}
unknown_domains = set(inc["mit_risk_domain"].dropna()) - set(GROUP_OF_DOMAIN)
assert not unknown_domains, f"MIT has a new risk domain: {unknown_domains}. Decide its incident type and add it above."

model_ids = set(labels.loc[labels["label"].eq("model"), "incident_id"])
keep = ["incident_id", "date", "year", "title", "mit_risk_domain", "mit_risk_subdomain", "report_count"]
model_rows = mentioned[mentioned["incident_id"].isin(model_ids)].merge(inc[keep], on="incident_id", how="left")
model_rows["group_mit"] = model_rows["mit_risk_domain"].map(GROUP_OF_DOMAIN).fillna("No MIT class")
print("Model incidents:", model_rows["incident_id"].nunique(), "| rows (incident and developer):", len(model_rows))
print("Without an MIT class:", model_rows.loc[model_rows["mit_risk_domain"].isna(), "incident_id"].nunique(), "incidents")

Model incidents: 318 | rows (incident and developer): 377
Without an MIT class: 66 incidents


## 9. Manual layer 2: the incidents that MIT has not classified

For a model incident without an MIT risk domain, the group assigned the closest MIT risk domain by hand, using the same domains MIT uses (file `manual/aiid_66_assigned_mit_domain.csv`). These types are the group's judgement and not an MIT classification, so the table keeps them in their own column and `type_source` says `Project group`. The notebook stops if a model incident without an MIT class has no row in the file, and writes the incidents to read to `processed/needs_domain.csv`.

In [7]:
assigned = pd.read_csv(MANUAL / "aiid_66_assigned_mit_domain.csv")
assert assigned["incident_id"].is_unique, "duplicated incident number in aiid_66_assigned_mit_domain.csv"
bad_domain = set(assigned["assigned_domain"]) - set(GROUP_OF_DOMAIN)
assert not bad_domain, f"assigned_domain is not an MIT domain: {bad_domain}"

unclassified_ids = set(model_rows.loc[model_rows["mit_risk_domain"].isna(), "incident_id"])
no_domain = sorted(unclassified_ids - set(assigned["incident_id"]))
if no_domain:
    (inc[inc["incident_id"].isin(no_domain)][["incident_id", "date", "title", "description", "developer", "deployer",
                                              "implicated_systems"]]
     .to_csv(OUT / "needs_domain.csv", index=False, encoding="utf-8"))
    raise SystemExit(f"{len(no_domain)} model incidents have no MIT class and no assigned domain. "
                     f"Read them in processed/needs_domain.csv and add rows to manual/aiid_66_assigned_mit_domain.csv.")

classified_by_mit = sorted(set(assigned["incident_id"]) & set(model_rows.loc[model_rows["mit_risk_domain"].notna(), "incident_id"]))
if classified_by_mit:
    print("MIT has since classified these incidents. Remove them from the manual file so that MIT's own class is used:",
          classified_by_mit)

table = model_rows.merge(assigned[["incident_id", "assigned_domain"]], on="incident_id", how="left")
table.loc[table["mit_risk_domain"].notna(), "assigned_domain"] = np.nan  # MIT's own class always wins
table["type_source"] = np.where(table["mit_risk_domain"].notna(), "MIT", "Project group")
table["domain_shown"] = table["mit_risk_domain"].fillna(table["assigned_domain"])
table["group"] = table["domain_shown"].map(GROUP_OF_DOMAIN)

columns = ["incident_id", "date", "year", "developer", "found_in", "title", "mit_risk_domain", "mit_risk_subdomain",
           "group_mit", "report_count", "assigned_domain", "type_source", "domain_shown", "group"]
table["date"] = table["date"].dt.strftime("%Y-%m-%d")
order = {name: k for k, name in enumerate(DEVELOPER_PATTERNS)}   # developers in the order of the list in Section 6
table = (table[columns].assign(_o=table["developer"].map(order))
         .sort_values(["incident_id", "_o"]).drop(columns="_o").reset_index(drop=True))
table.head()

,incident_id,date,year,developer,found_in,title,mit_risk_domain,mit_risk_subdomain,group_mit,report_count,assigned_domain,type_source,domain_shown,group
0,85,2020-10-09,2020,OpenAI,developer field,"AI attempts to ease fear of robots, blurts out...","AI system safety, failures, and limitations",Lack of capability or robustness,Wrong output or failure,1,NaN,MIT,"AI system safety, failures, and limitations",Wrong output or failure
1,102,2020-03-23,2020,Google,developer field,Personal voice assistants struggle with black ...,Discrimination and Toxicity,Unequal performance across groups,Harm to users or society,2,NaN,MIT,Discrimination and Toxicity,Harm to users or society
2,118,2020-08-06,2020,OpenAI,developer field,OpenAI's GPT-3 Associated Muslims with Violence,Discrimination and Toxicity,Unfair discrimination and misrepresentation,Harm to users or society,3,NaN,MIT,Discrimination and Toxicity,Harm to users or society
3,120,2020-09-01,2020,OpenAI,developer field,Philosophy AI Allegedly Used To Generate Mixtu...,Misinformation,False or misleading information,Wrong output or failure,1,NaN,MIT,Misinformation,Wrong output or failure
4,177,2022-04-19,2022,Google,developer field,Google’s Assistive Writing Feature Provided Al...,"AI system safety, failures, and limitations",Lack of capability or robustness,Wrong output or failure,5,NaN,MIT,"AI system safety, failures, and limitations",Wrong output or failure


## 10. Validate

These checks stop the notebook if the table breaks a rule the page relies on.

In [8]:
problems = []
if table.duplicated(["incident_id", "developer"]).any():
    problems.append("an incident and developer pair appears twice")
if table["group"].isna().any():
    problems.append("a row has no incident type")
if not set(table["developer"]) <= set(DEVELOPER_PATTERNS):
    problems.append("a developer outside the eleven")
if table["year"].min() < FIRST_YEAR:
    problems.append("an incident before the first year")
if pd.to_datetime(table["date"]).max() > pd.Timestamp(export_date) + pd.Timedelta(days=1):
    problems.append("an incident dated after the export")
if table["report_count"].isna().any():
    problems.append("a missing report count")
if table.groupby("incident_id")["title"].nunique().max() > 1:
    problems.append("one incident with two titles")
if not table["type_source"].eq("Project group").eq(table["mit_risk_domain"].isna()).all():
    problems.append("type_source does not follow the MIT class")
print("Problems:", problems if problems else "none")
assert not problems, problems

per_incident = table.drop_duplicates("incident_id")
print()
print("Model incidents:", len(per_incident), "| rows:", len(table))
print("Type by source (incidents):")
print(pd.crosstab(per_incident["group"], per_incident["type_source"], margins=True).to_string())
print()
print("Incidents of the last year by source of the type:")
last = per_incident[per_incident["year"].eq(per_incident["year"].max())]
print(last["type_source"].value_counts().to_string())

Problems: none

Model incidents: 318 | rows: 377
Type by source (incidents):
type_source               MIT  Project group  All
group                                            
Data and security          31              7   38
Harm to users or society   61              4   65
Misuse by others           54             24   78
Wrong output or failure   106             31  137
All                       252             66  318

Incidents of the last year by source of the type:
type_source
Project group    48
MIT              28


## 11. Not counted: incidents that name a developer but are not model incidents

For the record, how the incidents that name one of the eleven developers are split by label. Only `model` is on the page. The other labels are explained in Section 2.2.

In [9]:
print(labels[labels["incident_id"].isin(named_ids)]["label"].value_counts().to_string())
not_counted = (mentioned[~mentioned["incident_id"].isin(model_ids)]
               .merge(labels[["incident_id", "label", "label_detail"]], on="incident_id")
               .merge(inc[["incident_id", "date", "title"]], on="incident_id"))
not_counted["date"] = not_counted["date"].dt.strftime("%Y-%m-%d")
not_counted = not_counted[["incident_id", "date", "title", "developer", "label", "label_detail"]]
not_counted.sort_values(["label", "incident_id"]).head(8)

label
model         318
platform       61
venue_only     16
operations      5


,incident_id,date,title,developer,label,label_detail
26,372,2022-07-22,Users Reported Security Issues with Google Pix...,Google,operations,Not model behaviour (operations)
35,450,2021-11-01,Kenyan Data Annotators Reported Psychological ...,OpenAI,operations,Not model behaviour (operations)
76,1418,2026-02-27,Meta AI Smart Glasses Reportedly Routed Intima...,Meta,operations,Not model behaviour (operations)
80,1584,2026-05-20,Meta's AI-Assisted Layoff Process Allegedly Di...,Meta,operations,Not model behaviour (operations)
81,1677,2025-08-01,xAI Reportedly Operated Unpermitted Gas Turbin...,xAI,operations,Not model behaviour (operations)
0,81,2020-10-21,"Researchers find evidence of racial, gender, a...",Google,platform,Platform or non-LLM algorithm
1,82,2020-10-21,#LekkiMassacre: Why Facebook labelled content ...,Meta,platform,Platform algorithm (Facebook only)
2,83,2020-10-22,AI Spam Filters Allegedly Block Legitimate Ema...,Google,platform,Platform or non-LLM algorithm


## 12. Save

In [10]:
table.to_csv(OUT / "aiid_model_incidents_by_developer.csv", index=False, encoding="utf-8")
not_counted.sort_values(["label", "incident_id"]).to_csv(OUT / "aiid_not_counted_incidents.csv", index=False,
                                                          encoding="utf-8")
print("saved", OUT / "aiid_model_incidents_by_developer.csv", len(table), "rows")
print("saved", OUT / "aiid_not_counted_incidents.csv", not_counted["incident_id"].nunique(), "incidents")

saved d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\aiid_model_incidents\processed\aiid_model_incidents_by_developer.csv 377 rows
saved d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\aiid_model_incidents\processed\aiid_not_counted_incidents.csv 82 incidents


## How to update, and what to do if the download is blocked

**Automatic.** Set `REFRESH = True` in Section 3 and run all cells. The notebook reads the snapshots page, downloads the newest `AIID_Excel_Export-YYYYMMDD.xlsx` into `raw/` and records it in `raw/retrieval_log.json`. About 10 seconds.

**Manual download (use this if you see `Download of the export failed`).**

1. Open https://incidentdatabase.ai/research/snapshots/ in a browser and download the newest Excel export.
2. Save it as `manual\AIID_Excel_Export-YYYYMMDD.xlsx` (inside this folder). Keep the name.
3. Run the notebook again. It copies the newest file into `raw/` and logs it as `copied from manual folder`.

**New incidents need a reading.** A newer export has new incidents. The notebook does not guess about them.

1. If it stops with `... have no label`, open `processed/needs_label.csv`, read each incident and add a row to `manual/aiid_incident_labels_400.csv` (`model`, `platform`, `operations` or `venue_only`).
2. If it stops with `... no MIT class and no assigned domain`, open `processed/needs_domain.csv`, choose the closest MIT risk domain by comparing with incidents MIT classified, and add a row to `manual/aiid_66_assigned_mit_domain.csv` with `second_check` set to `pending`.
3. Run again. The numbers on the page will change and will no longer match the report text that quotes 318 incidents.

**Update check.** Open `raw/retrieval_log.json` (`how` is `downloaded` or `copied from manual folder`) and `processed/data_currency.csv`.

## Data currency: how up to date is this data?

In [11]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10],
        "data_covers_to": str(covers_to)[:10],
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


write_currency(
    "AI Incident Database model incidents by developer (aiid_model_incidents_by_developer.csv)",
    table["date"].min(), table["date"].max(),
    f"weekly export dated {export_date} (file in raw/, {log['how']}, retrieved {log['retrieved_utc']}, see raw/retrieval_log.json). "
    "The latest incident date is the date the harm occurred and is usually earlier than the export day, because incidents are added "
    "after reports appear. Incident types of incidents without an MIT class are the group's reading, see manual/",
    "The database started before 2020. This table keeps incidents dated 2020 or later. Database start date not recorded in our notes.",
    "A new export every Monday")

dataset                  AI Incident Database model incidents by developer (aiid_model_incidents_by_developer.csv)
extracted_utc            2026-11-14T02:21:29Z
data_covers_from         2020-03-23
data_covers_to           2026-10-03
snapshot_note            weekly export dated 2026-10-12 (file in raw/, downloaded, retrieved 2026-10-17T23:10:45Z, see raw/retrieval_log.json). The latest incident date is the date the harm occurred and is usually earlier than the export day, because incidents are added after reports appear. Incident types of incidents without an MIT class are the group's reading, see manual/
source_origin_note       The database started before 2020. This table keeps incidents dated 2020 or later. Database start date not recorded in our notes.
source_update_frequency  A new export every Monday


In [12]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 2.1 seconds
Files written:
  processed\aiid_model_incidents_by_developer.csv  (106,117 bytes)
  processed\aiid_not_counted_incidents.csv  (14,143 bytes)
  processed\data_currency.csv  (757 bytes)


## 13. Copy the table into the story data folder (by hand)

The notebook does not write into the story's `data/` folder. After a run, copy the file listed below from `processed/` into `../../data/` by hand. The page reads only that copy. The cell below only prints. It shows which file to copy, its number of rows and whether the copy already in `data/` has the same content.

In [13]:
TO_COPY = ["aiid_model_incidents_by_developer.csv"]   # the other files stay here as a record of the run
for name in TO_COPY:
    made = OUT / name
    rows_made = len(pd.read_csv(made))
    kept = STORY_DATA / name
    if not kept.exists():
        state = "not in the story data folder yet. Copy it."
    elif pd.read_csv(kept, keep_default_na=False).equals(pd.read_csv(made, keep_default_na=False)):
        state = "the copy in the story data folder has the same content. Nothing to do."
    else:
        state = "the copy in the story data folder is DIFFERENT. Copy the new file over it."
    print(f"{name}: {rows_made} rows. From {made.relative_to(HERE)} to {kept}. Status: {state}")

aiid_model_incidents_by_developer.csv: 377 rows. From processed\aiid_model_incidents_by_developer.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\data\aiid_model_incidents_by_developer.csv. Status: the copy in the story data folder has the same content. Nothing to do.


## 14. Run time and limits

**Time to extract.** One file of about 2.6 MB. The download takes a few seconds and the rest about 10 seconds. The last cells print the measured time.

**Limits and known risks.**

1. The incident database is a record of what was reported and what its editors chose to add. Counts follow reporting and media attention, not risk.
2. The eleven developers are found by text patterns in two fields. A developer typed in an unusual way is missed, and an incident that names a developer only inside a long list is counted for it.
3. The label (`model` or not) and the type of the incidents without an MIT class are the group's reading with AI assistance. A second group member has not checked them. Roughly 13 incidents about AI agents that test cyber attacks have no direct MIT precedent and are the most open to argument.
4. MIT classifies with a lag, so the share of incidents without an MIT class is highest among the newest incidents. When MIT classifies one of them, its own class replaces the group's reading (Section 9 prints those incidents).
5. An incident counts once for each developer it names, so the sum over developers is larger than the number of incidents.

## 15. References

1. Responsible AI Collaborative, AI Incident Database. https://incidentdatabase.ai
2. AI Incident Database, snapshots and weekly export. https://incidentdatabase.ai/research/snapshots/
3. AI Incident Database, terms of use. https://incidentdatabase.ai/terms-of-use
4. Creative Commons, Attribution-ShareAlike 4.0 International. https://creativecommons.org/licenses/by-sa/4.0/
5. MIT AI Risk Repository (the source of the risk domains). https://airisk.mit.edu